<a href="https://colab.research.google.com/github/harrisonritz/ccn_dynamics_first/blob/main/03b_switching_lds_fmri_hcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CCN Tutorial · Notebook 3
## Switching LDS on **HCP task fMRI**: do the discrete regimes track the task?

Notebook 3 fits the switching LDS (SLDS) to the **HCP `LANGUAGE` task**. Subjects alternated between listening to short **stories** and solving auditory **math** problems — parcellated to the Schaefer-100 / Yeo-17 atlas by the pipeline in `src/` (see `src/README.md`). Each recording is one run of $T=316$ TRs at TR $=0.72$ s; the dataset holds **43 subjects × 2 runs (LR, RL) = 86 runs**.

The task has a known **block design** (a story/math *boxcar*). That gives us something the resting-state notebook lacked: **ground truth about when the cognitive context changes.** 

> **Does an SLDS, fit with no knowledge of the task, place its discrete-regime boundaries where the story↔math blocks change?**


1. **§2–3** — load the parcellated tensors with `NPZ.jl`, one run per trial, holding out every run of the last `N_HELDOUT_SUBJECTS` subjects; reduce the 100 parcels to `N_PC` observation dimensions, either by unsupervised PCA or by a task-informed story-vs-math GED basis (`REPRESENTATION = :ged`).
2. **§4–5** — fit an SLDS over a **user-set grid** of discrete states $K$ × latent dimensions, and compare models by **held-out ELBO** on the unseen subjects (add $K=1$ to the grid for the single-regime LDS baseline).
3. **§6** — pick one fitted model and read its parameters: the discrete transitions, each regime's dynamics $A_k$ and offsets $b_k,d_k$, and whether the regimes separate at all.
4. **§7** — **posterior predictive checks**: predicted vs. observed carpets and single-channel traces carrying estimation and prediction bands. Does the model reproduce the BOLD signal in the first place?
5. **§8** — the **continuous** latent averaged around story↔math switches, against a circular-shift null. Is $\hat x_t$ locked to the block boundaries?
6. **§9** — the **discrete** regimes laid over the task boxcar, scored by occupancy and by mutual information / NMI against the same kind of null. Is $\hat z_t$?
7. **§10** — each regime's mean and connectivity projected back to the 100 parcels and rendered on the fsLR-32k surface.

Data and atlas files are downloaded in §0; every knob you might want to sweep lives in one cell in §1.


## 0. Setup

Same environment as Notebook 3, plus **`NPZ.jl`** to read the parcellation pipeline's `.npz`. In Colab set **Runtime → Change runtime type → Julia** first, then run this cell once (a few minutes to precompile).


In [ ]:
using Pkg
for p in ["MatrixEquations", "StableRNGs", "Plots", "NPZ", "PyCall", "Conda"]
    try
        Base.require(Main, Symbol(p));
    catch
        Pkg.add(p);
    end
end
Pkg.build("PyCall")

# install StateSpaceDynamics.jl
Pkg.add(url="https://github.com/depasquale-lab/StateSpaceDynamics.jl.git", rev="CCN2026")
using StateSpaceDynamics, LinearAlgebra, Statistics, Random, StableRNGs, NPZ, Plots, PyCall, Conda, Downloads
const SSD = StateSpaceDynamics
println("ready — StateSpaceDynamics v", pkgversion(SSD))

# python for the surface plots. it only ever runs OUT OF PROCESS (see the renderer cell):
# nilearn pulls in matplotlib/scipy/scikit-learn, and loading their native libraries next to
# the ones julia already holds (libfreetype via GR, libgfortran/libgomp via BLAS) takes the
# kernel down with it. so everything here talks to python through subprocesses instead.
PYPKGS = ["numpy", "nibabel", "matplotlib", "nilearn"]
PYDEPS = joinpath(homedir(), ".ccn_pydeps")   # last-resort private package dir

function can_import(exe, mods)
    try
        success(`$exe -c "import $(join(mods, ", "))"`)
    catch
        false
    end
end

function pydeps_on_path!()
    occursin(PYDEPS, get(ENV, "PYTHONPATH", "")) && return
    ENV["PYTHONPATH"] = isempty(get(ENV, "PYTHONPATH", "")) ? PYDEPS : PYDEPS * ":" * ENV["PYTHONPATH"]
end

# the interpreter: the first one that can already import numpy+matplotlib. on colab the julia
# image's conda python has a matplotlib that will not load (libraqm.so.0: undefined symbol)
# while the system python3 is fine; locally PyCall's python is usually the one with everything.
PYCANDS = [PyCall.python]
let s = Sys.which("python3")
    s === nothing || push!(PYCANDS, s)
end
PYEXE = PYCANDS[something(findfirst(c -> can_import(c, ["numpy", "matplotlib"]), PYCANDS), 1)]

function pip(args::Vector{String})            # never throws; true if pip exited cleanly
    base = `$PYEXE -m pip install --quiet`
    ok = try
        success(`$base $args`)
    catch
        ;
        false
    end
    ok || (ok = try
        success(`$base --break-system-packages $args`)
    catch
        ;
        false
    end)
    return ok
end

function ensure_python_pkgs(pkgs)
    isdir(PYDEPS) && pydeps_on_path!()
    for p in pkgs
        can_import(PYEXE, [p]) && continue
        println("installing $p into $PYEXE");
        flush(stdout)
        pip(["--upgrade", p])
        can_import(PYEXE, [p]) && continue
        # installed but not loadable: a conda build against a broken system library.
        # overwrite just that package with a PyPI wheel, which bundles its own libraries.
        println("  $p is installed but won't import — replacing it with a self-contained wheel");
        flush(stdout)
        pip(vcat(["--force-reinstall", "--no-deps", "--only-binary=:all:"],
            p == "matplotlib" ? ["matplotlib", "pillow"] : [p]))
        can_import(PYEXE, [p]) && continue
        # last resort: our own copy, first on PYTHONPATH, shadowing whatever is broken
        println("  installing $p into $PYDEPS and putting it first on PYTHONPATH");
        flush(stdout)
        pip(["--target", PYDEPS, "--upgrade", "--only-binary=:all:", "--no-deps", p])
        pydeps_on_path!()
        can_import(PYEXE, [p]) || error("could not import $p with $PYEXE — see the pip output above")
    end
    report = """
    import importlib
    for p in [$(join(map(p -> "'$p'", pkgs), ", "))]:
        m = importlib.import_module(p)
        print('  %-11s v%s   %s' % (p, m.__version__, m.__file__))
    """
    println("python ready: $PYEXE")
    run(`$PYEXE -c $report`)
end
ensure_python_pkgs(PYPKGS)

# setup plotting backend
gr();
ENV["GKSwstype"] = "100"                 # headless plotting backend
default(framestyle=:box, grid=false, label="")


# print confirmation
println("--------------------------------\n Everything is ready! You can now use StateSpaceDynamics.jl and its dependencies.\n--------------------------------")

In [ ]:
# --- download assets ---
# Schaefer-100/17 fsLR dlabel + fsLR-32k inflated surfaces + the parcellated HCP tensors.
#
#   :gcloud → everything in one zip from a public Google Cloud Storage bucket (default)
#   :github → per-file from the original public atlas/surface repos; this covers the atlas
#             files only, so the parcellated HCP tensors still come from the bucket.
using p7zip_jll                             # stdlib: the 7z binary that ships with Julia
ASSET_SOURCE = :gcloud

GCS_ZIP_URI = "gs://ccn2026-dynamics/HCP_data.zip"        # public object, fetched over https below
gcs_https(uri) = replace(uri, r"^gs://" => "https://storage.googleapis.com/")

DERIV = "derivatives"
ASSETS = [                                  # (local path, per-file URL or "" if the zip is the only source)
    ("$DERIV/schaefer100_17net_fslr32k.dlabel.nii",
        "https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/HCP/fslr32k/cifti/Schaefer2018_100Parcels_17Networks_order.dlabel.nii"),
    ("$DERIV/fs_LR.32k.L.inflated.surf.gii",
        "https://raw.githubusercontent.com/DiedrichsenLab/fs_LR_32/master/fs_LR.32k.L.inflated.surf.gii"),
    ("$DERIV/fs_LR.32k.R.inflated.surf.gii",
        "https://raw.githubusercontent.com/DiedrichsenLab/fs_LR_32/master/fs_LR.32k.R.inflated.surf.gii"),
    ("$DERIV/hcp_language_schaefer100.npz", ""),
    ("$DERIV/hcp_language_schaefer100_parcels.csv", ""),
    ("$DERIV/hcp_language_schaefer100_metadata.json", "")]

mkpath(DERIV)                               # Colab starts in /content with no derivatives/
have(path) = isfile(path) && filesize(path) > 0
is_markup(path) = occursin(r"<\?xml|<html|<!doctype"i, String(open(io -> read(io, 512), path)))

function fetch_gcloud()                     # one zip → unpacked into derivatives/
    zip = joinpath(DERIV, basename(GCS_ZIP_URI))
    println("downloading ", gcs_https(GCS_ZIP_URI))
    Downloads.download(gcs_https(GCS_ZIP_URI), zip)
    # an object that is not publicly readable answers with an XML error page, not the archive
    is_markup(zip) && (rm(zip); error("$GCS_ZIP_URI came back as an error page, not the zip — the object " *
                                      "must be readable by allUsers (Storage Object Viewer)"))
    run(pipeline(`$(p7zip()) x -y -o$DERIV $zip`, devnull))
    rm(zip)
    println("unpacked ", basename(GCS_ZIP_URI), " into $DERIV/")
end

function fetch_github()                     # per-file, straight from the public atlas/surface repos
    for (path, url) in ASSETS
        (have(path) || isempty(url)) && continue
        println("downloading $path")
        Downloads.download(url, path)
    end
end

println("fetching assets from $ASSET_SOURCE ...")
if all(have, first.(ASSETS))
    println("all assets already present")
elseif ASSET_SOURCE == :gcloud
    fetch_gcloud()
else
    fetch_github()
    all(have, first.(ASSETS)) || fetch_gcloud()   # the HCP tensors only live in the bucket
end
for (path, _) in ASSETS
    have(path) || error("missing $path — re-run this cell with ASSET_SOURCE = :gcloud")
end
println("assets ready in $DERIV/")

DLABEL, SURF_L, SURF_R, HPC_DATA, PARCELS_CSV, META_JSON = first.(ASSETS);

In [ ]:
# the surface renderer — the same validated nilearn code, run OUT OF PROCESS as a script.
# importing it here would load matplotlib/scipy/scikit-learn's native libraries alongside the
# copies julia already has and kill the kernel, so the parcel values go out over the command
# line and the finished png comes back through a temp file.
using Base64
PLOT_PY = joinpath(DERIV, "plot_parcel.py")
write(
    PLOT_PY,
    raw"""
import sys, numpy as np, nibabel as nib, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from nilearn import plotting

dlabel, surf_l, surf_r, out, title, cmap, sym, set_vmax, vals = sys.argv[1:10]
vals = np.array([float(v) for v in vals.split(',')])
sym = (sym == 'true'); set_vmax = float(set_vmax)

_img = nib.load(dlabel); _lab = np.asarray(_img.get_fdata()).astype(int).ravel()
_bm  = _img.header.get_axis(1); _NV = 32492
_vp  = {'CORTEX_LEFT': np.zeros(_NV, int), 'CORTEX_RIGHT': np.zeros(_NV, int)}
for _name, _sl, _ in _bm.iter_structures():
    _k = _name.replace('CIFTI_STRUCTURE_', '')
    if _k in _vp: _vp[_k][_bm.vertex[_sl]] = _lab[_sl]
_mesh = {'CORTEX_LEFT': nib.load(surf_l), 'CORTEX_RIGHT': nib.load(surf_r)}

def _vertmap(vals, hemi):
    pl = _vp[hemi]; out = np.full(_NV, np.nan); m = pl > 0
    out[m] = np.asarray(vals, float)[pl[m] - 1]; return out

vmax = float(np.nanmax(np.abs(vals)))
vmin = (-vmax if sym else float(np.nanmin(vals))); vmax = (vmax if sym else float(np.nanmax(vals)))
if set_vmax != 0: vmax = float(set_vmax); vmin = (-vmax if sym else float(np.nanmin(vals)))
fig, axes = plt.subplots(2, 2, subplot_kw={'projection': '3d'}, figsize=(8, 6))
order = [('CORTEX_LEFT','left','lateral'), ('CORTEX_RIGHT','right','lateral'),
         ('CORTEX_LEFT','left','medial'),  ('CORTEX_RIGHT','right','medial')]
for ax, (hemi, h, view) in zip(axes.ravel(), order):
    g = _mesh[hemi]; coords = g.darrays[0].data; faces = g.darrays[1].data
    plotting.plot_surf((coords, faces), _vertmap(vals, hemi), hemi=h, view=view, cmap=cmap,
                       vmin=vmin, vmax=vmax, axes=ax, colorbar=True, avg_method='mean')
fig.suptitle(title)
fig.savefig(out, format='png', dpi=100, bbox_inches='tight')
"""
)

function surfshow(vals; title="", cmap="coolwarm", sym=true, set_vmax=0)
    png = tempname() * ".png"
    run(`$PYEXE $PLOT_PY $DLABEL $SURF_L $SURF_R $png $title $cmap $sym $set_vmax $(join(collect(Float64, vals), ","))`)
    display("text/html", "<img style=\"max-width:100%\" src=\"data:image/png;base64,"
                         * base64encode(read(png)) * "\"/>")
    rm(png, force=true)
end
println("surface renderer ready (out of process)")

## 1. Parameters — *edit these*

Everything you might want to sweep lives here. `K_GRID` (discrete states) and `LATENT_GRID` (continuous latent dimensions) are the two axes of the model-comparison grid; **set either to a single-element list to sweep just the other axis** (e.g. `K_GRID = [2]`, `LATENT_GRID = [2, 4, 8]`).


In [ ]:
N_HELDOUT_SUBJECTS = 5               # last N subjects → validation (held out from fitting)
N_PC = 24              # PCA observation dimension fed to the SLDS
ZSCORE_PARCELS = true            # z-score each parcel (train stats) BEFORE PCA
REPRESENTATION = :zpca            # :zpca (unsupervised PCA) or :ged (story/math GED contrast).
K_GRID = [2]             # discrete regimes to compare   (edit to a subset)
LATENT_GRID = [2, 4, 8]   # latent dimensions to compare  (edit to a subset)
MAX_ITER = 200             # EM iterations per fit
PSTAY = 0.95            # initial self-transition prob; expected dwell ≈ 1/(1-PSTAY) TRs.
EMIS_JITTER = 0.1             # fallback emission-offset jitter (used only when !SEED_MEANS)
SEED_MEANS = false            # seed each regime's emission offset dₖ from the HRF-lagged
HRF_LAG_S = 5.0             # hemodynamic lag (s) applied to the boxcar when SCORING ẑ↔task
# (BOLD peaks ~4–6 s after the stimulus); set 0.0 to disable.
SEED = 99
Random.seed!(SEED)                   # the SLDS E-step is stochastic; seed for reproducibility
println("grid: K ∈ $K_GRID  ×  latent ∈ $LATENT_GRID   → ",
        length(K_GRID) * length(LATENT_GRID), " SLDS fits")

## 2. Load the HCP dataset

`NPZ.jl` reads the pipeline's tensors directly (see the key table in `src/README.md`). The pieces we need:

- **`data`** — `float32 (parcels=100, TRs=316, runs=48)`, already cleaned and per-run z-scored.
- **`boxcar`** — `int8 (TRs, 2, runs)`; **column 1 = story, column 2 = math** (a TR is *rest/fixation* when both are 0). It is stored **per run** because HCP's math blocks are adaptive, so we always read each run's own boxcar.
- **`subject_ids`** / **`slab_run_ids`** — which subject and which run (0 = LR, 1 = RL) each of the 48 slabs is.

Each run is one **trial** for the SLDS — exactly how the data is laid out (no concatenation across the LR/RL seam). We hold out **every run of the last `N_HELDOUT_SUBJECTS` subjects** for validation.


In [ ]:
scalar(x) = x isa AbstractArray ? x[] : x        # NPZ stores scalars as 0-d arrays

isfile(HPC_DATA) || error("Data not found at $HPC_DATA — build it with src/ or fix HPC_DATA")
npz = npzread(HPC_DATA)
X = Float64.(npz["data"])                   # parcels × TRs × runs
boxcar = npz["boxcar"]                            # TRs × 2 × runs  (col1 story, col2 math)
subj = vec(npz["subject_ids"])                 # run → HCP subject id
runcode = vec(npz["slab_run_ids"])               # run → 0 LR / 1 RL
TRsec = scalar(npz["tr"])
P, T, E = size(X)

# --- train / validation split BY SUBJECT (last N subjects held out) ---
subj_order = unique(subj)                                        # subjects in slab order
val_ids = Set(subj_order[(end-N_HELDOUT_SUBJECTS+1):end])
is_val = [s in val_ids for s in subj]                        # per-run mask
train_ep = findall(.!is_val)
val_ep = findall(is_val)

# --- per-run observation matrices (parcels × time) and task-condition labels ---
# condition code per TR:  1 = story, 2 = math, 0 = rest/fixation
cond(e) = [boxcar[t, 1, e] == 1 ? 1 : boxcar[t, 2, e] == 1 ? 2 : 0 for t in 1:T]

Xtr_parcels = [X[:, :, e] for e in train_ep]
Xva_parcels = [X[:, :, e] for e in val_ep]
cond_tr = [cond(e) for e in train_ep]
cond_va = [cond(e) for e in val_ep]
run_va = runcode[val_ep]                                    # LR/RL of each held-out run

# --- hemodynamic lag: BOLD (hence ẑ) trails the stimulus ~4–6 s; used for seeding AND scoring ---
HRF_LAG_TR = HRF_LAG_S > 0 ? round(Int, HRF_LAG_S / TRsec) : 0
hrf_shift(c) = HRF_LAG_TR == 0 ? c : vcat(fill(0, HRF_LAG_TR), c)[1:length(c)]  # delay boxcar by lag
cond_tr_lag = [hrf_shift(c) for c in cond_tr]
cond_va_lag = [hrf_shift(c) for c in cond_va]

println("subjects: $(length(subj_order))  → train $(length(subj_order) - N_HELDOUT_SUBJECTS), ",
    "val $N_HELDOUT_SUBJECTS  (", join(sort(collect(val_ids)), ", "), ")")
println("runs (trials): train $(length(train_ep)), val $(length(val_ep));  each $T TRs @ $(round(TRsec; digits=2))s")
println("task mix (all runs):  story ", round(100mean(boxcar[:, 1, :]); digits=1), "%  ",
    "math ", round(100mean(boxcar[:, 2, :]); digits=1), "%  ",
    "rest ", round(100mean((boxcar[:, 1, :] .== 0) .& (boxcar[:, 2, :] .== 0)); digits=1), "%",
)

# --- how long the task stays put between switches ---
# a block is a maximal stretch of one condition.  The block a run *ends* on is cut off by the
# scan rather than by a switch, so its length is censored and it is dropped; the block a run
# opens on is kept, since every run here starts on a full-length story block (36-40 TR).
function block_lengths(c, code)
    L, t, Tn = Int[], 1, length(c)
    while t <= Tn
        j = t
        while j < Tn && c[j+1] == c[t]
            j += 1
        end
        (c[t] == code && j < Tn) && push!(L, j - t + 1)
        t = j + 1
    end
    L
end

println("block lengths (time between switches, pooled over all runs; run-final blocks dropped as censored):")
for (code, nm) in zip((1, 2, 0), ("story", "math ", "rest "))
    L = reduce(vcat, block_lengths(cond(e), code) for e in 1:E)
    isempty(L) && continue
    q = quantile(L, [0.25, 0.5, 0.75])
    println("  ", nm, "  n = ", lpad(length(L), 4),
        "   Q1 / median / Q3 = ", join(round.(q; digits=1), " / "), " TR",
        "  = ", join(round.(q .* TRsec; digits=1), " / "), " s",
        "   (range ", minimum(L), "-", maximum(L), " TR)")
end

### A first look at the data

Before any modelling, three quick views of what we just loaded:

1. **parcel × TR heatmaps** for a few example runs, with the story (blue) / math (orange) blocks drawn as a ribbon above — the block structure should already be visible by eye in some parcels;
2. **condition-average parcel maps** — mean BOLD over all story TRs and all math TRs (HRF-lagged), plus their difference, rendered on the fsLR-32k surface with the `surfshow` helper from §0;
3. the **leading PCA topographies** of the training runs — the spatial patterns that the `N_PC`-dimensional observation space of §3 is built from.

Everything here is display-only: parcels are z-scored *per run* so runs are comparable in a heatmap. The model's own standardization and PCA are fit on the training runs in §3.

In [ ]:
# ---- a first look at the data (display only; nothing here feeds the model) ----
# the npz ships parcels on very different scales (std ≈ 13-90), so z-score each parcel
# within run for display; §3 does its own standardization on TRAIN statistics.
zrun(Y) = (Y .- mean(Y, dims=2)) ./ (std(Y, dims=2) .+ 1e-8)
Xz = cat([zrun(X[:, :, e]) for e in 1:E]...; dims=3)      # parcels × TRs × runs
cond_all = [hrf_shift(cond(e)) for e in 1:E]              # HRF-lagged condition per TR, every run

# --- 1. parcel × TR heatmaps for a few example runs -----------------------------
N_EXAMPLE = 3
example_ep = [findfirst(==(s), subj) for s in subj_order[1:min(N_EXAMPLE, length(subj_order))]]  # first run per subject

function cond_ribbon(c; title="")                          # thin story/math strip above a heatmap
    plt = plot(legend=false, framestyle=:none, title=title, titlefontsize=9,
        xlim=(0.5, length(c) + 0.5), ylim=(0, 1))
    t = 1
    while t <= length(c)
        j = t
        while j < length(c) && c[j+1] == c[t]
            j += 1
        end
        c[t] == 1 && vspan!(plt, [t - 0.5, j + 0.5], c=:dodgerblue, alpha=0.6)
        c[t] == 2 && vspan!(plt, [t - 0.5, j + 0.5], c=:orange, alpha=0.6)
        t = j + 1
    end
    plt
end

ribbons = [cond_ribbon(cond_all[e];
    title="sub $(subj[e]) · $(runcode[e] == 0 ? "LR" : "RL")  [story=blue, math=orange]")
           for e in example_ep]
heatmaps = [heatmap(Xz[:, :, e], c=:vik, clims=(-3, 3), colorbar=false,
    xlabel="TR", ylabel="parcel", titlefontsize=9) for e in example_ep]
display(plot(ribbons..., heatmaps...;
    layout=grid(2, length(example_ep), heights=[0.08, 0.92]),
    size=(340 * length(example_ep), 360)))

# --- 2. condition-average parcel maps (story, math, difference) -----------------
# per-run condition mean of the z-scored parcels, then averaged over all runs
condmean(code) = vec(mean(reduce(hcat,
        [mean(Xz[:, findall(==(code), cond_all[e]), e], dims=2) for e in 1:E]), dims=2))
story_map, math_map = condmean(1), condmean(2)
vmx = maximum(abs, vcat(story_map, math_map))
surfshow(story_map; title="mean z(BOLD) · STORY blocks (HRF-lagged)", set_vmax=vmx)
surfshow(math_map; title="mean z(BOLD) · MATH blocks (HRF-lagged)", set_vmax=vmx)
# surfshow(story_map .- math_map; title="story − math")

# --- 3. leading PCA topographies (train runs only) ------------------------------
N_SHOW_PC = 1
Xtr_z = reduce(hcat, [Xz[:, :, e] for e in train_ep])      # parcels × Σtime (train)
Fsvd = svd(Xtr_z .- mean(Xtr_z, dims=2))
ve = Fsvd.S .^ 2 ./ sum(Fsvd.S .^ 2)
println("variance explained: PC1-$N_SHOW_PC = ", round.(100 .* ve[1:N_SHOW_PC]; digits=1),
    "%   cumulative to N_PC = $N_PC: ", round(100 * sum(ve[1:N_PC]); digits=1), "%")
for n in 1:N_SHOW_PC
    u = Fsvd.U[:, n]
    u = u .* sign(u[argmax(abs.(u))])                      # PC sign is arbitrary — orient to strongest loading
    surfshow(u; title="PC $n loading  ($(round(100ve[n]; digits=1))% var)")
end

## 3. Observation space: PCA to `N_PC` components

Each SLDS regime carries a full $\text{obs}\times\text{obs}$ observation-noise matrix $R^{(k)}$, so 100 parcels would be heavy and slow across the grid. We reduce to `N_PC` principal components. To keep the held-out comparison honest, PCA is **fit on the training runs only** and the same loadings project the validation runs. Because the dataset ships un-standardized (per-parcel std ≈ 13–90), each **parcel** is first z-scored on the training runs when `ZSCORE_PARCELS` is on, so the PCA basis reflects the distributed story/math contrast rather than a handful of high-variance parcels. The resulting components are then z-scored too (train statistics) so every observation dimension is on the same scale. Set `REPRESENTATION = :ged` to instead use a **generalized eigendecomposition** of the story-vs-math covariance (fit on training runs only, HRF-lagged) — a task-informed basis that concentrates the contrast into the top components. We verified on this data that it sharpens *both* the mean and the **connectivity ($A$)** difference, not just amplitude. The spatial loadings $V$ let us project regimes back to parcels later if we wish.


In [ ]:
function fit_pca(trials, n_pc; zscore_parcels=true)
    Xall = reduce(hcat, trials)                       # parcels × Σtime
    pμ = mean(Xall, dims=2)                          # per-parcel mean (centering)
    pσ = zscore_parcels ? std(Xall, dims=2) .+ 1e-8 : ones(size(Xall, 1), 1)  # per-parcel scale
    Xs = (Xall .- pμ) ./ pσ                          # standardized parcels (centered only if !zscore)
    F = svd(Xs)                                     # thin SVD (parcels × time)
    V = F.U[:, 1:n_pc]                              # parcels × n_pc loadings
    sc = V' * Xs                                     # n_pc × Σtime training scores
    varexp = cumsum(F.S .^ 2) ./ sum(F.S .^ 2)
    sμ, sσ = mean(sc, dims=2), std(sc, dims=2) .+ 1e-8
    sμ, sσ = zeros(size(sμ)), ones(size(sσ))  # PCA scores are already standardized; override the empirical mean/std
    (V=V, pμ=pμ, pσ=pσ, sμ=sμ, sσ=sσ, varexp=varexp)
end

# GED: directions maximizing the story-vs-math covariance contrast (task-informed, Stage B).
# Fit on TRAINING frames only, HRF-lagged; take components from BOTH extremes of the spectrum.
function fit_ged(trials, condL, n_comp; zscore_parcels=true)
    Xall = reduce(hcat, trials)
    pμ = mean(Xall, dims=2)
    pσ = zscore_parcels ? std(Xall, dims=2) .+ 1e-8 : ones(size(Xall, 1), 1)
    Z = (Xall .- pμ) ./ pσ
    cc = reduce(vcat, condL)                         # per-frame condition (HRF-lagged)
    function classcov(lab)
        Zc = Z[:, cc .== lab];
        Zc = Zc .- mean(Zc, dims=2)
        Symmetric(Zc * Zc' ./ size(Zc, 2))
    end
    Cs, Cm = classcov(1), classcov(2)                  # story, math covariances
    Fg = eigen(Symmetric(Matrix(Cs)), Symmetric(Matrix(Cm)))  # ascending generalized eigenvalues
    npk = n_comp÷2
    sel = vcat(1:npk, (length(Fg.values)-(n_comp-npk)+1):length(Fg.values))  # both extremes
    # back to parcel space
    V = Fg.vectors[:, sel]
    V = V ./ sqrt.(sum(V .^ 2, dims=1))              # unit-norm loadings
    sc = V' * Z
    sμ, sσ = mean(sc, dims=2), std(sc, dims=2) .+ 1e-8
    (V=V, pμ=pμ, pσ=pσ, sμ=sμ, sσ=sσ, varexp=Float64[])  # GED doesn't have a variance-explained metric
end

project(p, Y) = (p.V' * ((Y .- p.pμ) ./ p.pσ) .- p.sμ) ./ p.sσ  # parcels×time → n_comp×time, z-scored

reducer = REPRESENTATION == :ged ?
          fit_ged(Xtr_parcels, cond_tr_lag, N_PC; zscore_parcels=ZSCORE_PARCELS) :
          fit_pca(Xtr_parcels, N_PC; zscore_parcels=ZSCORE_PARCELS)
Ytr = [project(reducer, Y) for Y in Xtr_parcels]  # each N_PC × 316
Yva = [project(reducer, Y) for Y in Xva_parcels]
obs_dim = N_PC
println("obs space: $P parcels → $N_PC ",
    REPRESENTATION == :ged ? "GED comps (story/math contrast)" : "PCA comps",
    ZSCORE_PARCELS ? "  (parcels z-scored)" : "  (parcels centered only)",
    REPRESENTATION == :ged ? "" :
    ";  train var explained = $(round(100 * reducer.varexp[N_PC]; digits=1))%")

## 4. SLDS recipe: LDS warm-start, robust fit, frozen inference

Identical machinery to Notebook 3, lifted to a **latent-dimension parameter** (so we can sweep it) and a **general latent size** (Notebook 3 hard-coded 2-D rotations; here each regime's dynamics matrix is a block of 2-D rotations at a distinct frequency/decay, padded for odd dimensions).

- `warmstart_lds(l)` fits one ordinary LDS per latent size — the seed for every regime at that size.
- `init_slds` warm-start each regime from that LDS's emission, spread the regimes across rotation frequencies, add weak inverse-Wishart priors
- With `SEED_MEANS`, each regime's emission offset $d_k$ is initialized from the HRF-lagged story/math means, so the regimes start **aligned to condition**; EM then differentiates each regime's mean ($b_k$) **and** connectivity ($A_k$) — the latter being the LDS's value-add over a GLM.
- `infer_slds` runs the variational E-step with **frozen** parameters and returns the ELBO and the regime responsibilities $\gamma_{k,t}=q(z_t=k)$ — used for held-out scoring and for decoding.


In [ ]:
rot(θ, r) = r * [cos(θ) -sin(θ); sin(θ) cos(θ)]

# an l×l dynamics matrix built from 2-D rotation blocks at angle θ, radius r
function rot_block(l, θ, r)
    A = zeros(l, l)
    for i in 1:2:(l-1)
        A[i:(i+1), i:(i+1)] .= rot(θ, r)
    end
    isodd(l) && (A[l, l] = r)
    A
end

# Covariance (IW) halves
Q_prior(l) = IWPrior(Ψ=Matrix(0.1I, l, l), ν=Float64(l + 5.0))
P0_prior(l) = IWPrior(Ψ=Matrix(0.1I, l, l), ν=Float64(l + 5.0))
R_prior(o) = IWPrior(Ψ=Matrix(0.1I, o, o), ν=Float64(o + 5.0))

# Mean (MN) halves
AB_prior(l) = MNPrior(M₀=hcat(Matrix(I, l, l), zeros(l)), Λ=Matrix(0.001I, l + 1, l + 1))
CD_prior(o, l) = MNPrior(M₀=zeros(o, l + 1), Λ=Matrix(0.001I, l + 1, l + 1))
x0_prior(l) = x0_mean_prior(zeros(l); κ₀=1.0)

function init_lds(l, o; seed=99)
    r = StableRNG(seed)

    LinearDynamicalSystem(
        GaussianStateModel(A=0.95Matrix(I, l, l), b=zeros(l), Q=Matrix(0.05I, l, l),
            x0=zeros(l), P0=Matrix(1.0I, l, l),
            AB_prior=AB_prior(l), Q_prior=Q_prior(l),
            x0_prior=x0_prior(l), P0_prior=P0_prior(l),
        ),
        GaussianObservationModel(C=randn(r, o, l), R=Matrix(0.3I, o, o), d=zeros(o),
            CD_prior=CD_prior(o, l), R_prior=R_prior(o),
        ),
    )
end

const LDS_CACHE = Dict{Int,Any}()                    # one warm-start LDS per latent dim

function warmstart_lds(l, o, Y)
    haskey(LDS_CACHE, l) && return LDS_CACHE[l]
    m = init_lds(l, o; seed=SEED)
    fit!(m, Y; max_iter=100, progress=true)
    LDS_CACHE[l] = m
end

# per-regime emission-offset seeds from the HRF-lagged story/math means (reduced obs space)
function seed_means(Y, condL, K, o)
    Yc = reduce(hcat, Y);
    cc = reduce(vcat, condL)
    conds = [1, 2, 0]                                  # story, math, rest (priority order)
    D = zeros(o, K)
    for k in 1:K
        idx = findall(==(conds[min(k, length(conds))]), cc)
        isempty(idx) || (D[:, k] = vec(mean(Yc[:, idx], dims=2)))
    end
    return D
end

function init_slds(K, l, o, lds; pstay=PSTAY, emis_jitter=EMIS_JITTER, d_seeds=nothing, seed=SEED)
    angs = K == 1 ? [2π/12.0] : collect(range(2π/45, -2π/45, length=K))   # spread of frequencies
    rads = K == 1 ? [0.95] : collect(range(0.80, 0.985, length=K))   # spread of decays
    Af, bf, Qf = lds.state_model.A, lds.state_model.b, lds.state_model.Q
    Cf, df, Rf = lds.obs_model.C, lds.obs_model.d, lds.obs_model.R
    x0f, P0f = lds.state_model.x0, lds.state_model.P0
    r = StableRNG(seed)
    dk(i) = d_seeds === nothing ? emis_jitter .* randn(r, o) : copy(d_seeds[:, i])
    println(d_seeds === nothing ? "emission offsets dₖ: random jitter" : "emission offsets dₖ: seeded from HRF-lagged story/math means")
    mode(i) = LinearDynamicalSystem(
        GaussianStateModel(
            A=0.5*rot_block(l, angs[i], rads[i]) .+ 0.5*Af, b=emis_jitter*randn(r, l), Q=copy(Qf),
            AB_prior=AB_prior(l), Q_prior=Q_prior(l),
            x0=x0f, P0=P0f,
            x0_prior=x0_prior(l), P0_prior=P0_prior(l),
        ),
        GaussianObservationModel(
            # C=copy(Cf), d=dk(i), R=copy(Rf),
            C=copy(Cf), d=zeros(o), R=copy(Rf),
            CD_prior=CD_prior(o, l), R_prior=R_prior(o),
        ),
        fit_bool=vcat(true, true, true, true, false, true) #[x0, P0, A&b&B, Q, C&d&D, R]
    )
    Π = fill((1 - pstay) / max(K - 1, 1), K, K);
    for k in 1:K
        Π[k, k] = K == 1 ? 1.0 : pstay;
    end
    return SLDS(A=Π, πₖ=fill(1.0/K, K), LDSs=[mode(i) for i in 1:K])
end

function fit_slds(K, l, Y, lds; d_seeds=nothing, max_iter=MAX_ITER)
    s = init_slds(K, l, obs_dim, lds; d_seeds=d_seeds)
    el = fit!(s, Y; max_iter=max_iter, progress=true)
    @assert all(isfinite, el)
    return s, el
end

println("helpers defined")

## 5. Model comparison: held-out ELBO across the grid

For each $(K,\text{latent})$ we fit the SLDS on the training runs and score it by **held-out ELBO per element** on the runs of the `N_HELDOUT_SUBJECTS` unseen subjects. $K=1$ is the single-regime LDS baseline, measured on the same scale. Because these are unseen *subjects* (not just unseen time), a rise in held-out ELBO with $K$ means the extra regimes generalize across people, not just memorize one recording.


In [ ]:
results = Matrix{Float64}(undef, length(K_GRID), length(LATENT_GRID))  # held-out ELBO / element
all_elbos = Matrix{Vector{Float64}}(undef, length(K_GRID), length(LATENT_GRID))  # training ELBO / iteration
fitted = Dict{Tuple{Int,Int},Any}()
Nva = sum(size(y, 2) for y in Yva)                                 # total held-out TRs
empty!(LDS_CACHE)

for (jl, l) in enumerate(LATENT_GRID)
    lds_ws = warmstart_lds(l, obs_dim, Ytr)                            # warm-start LDS for this latent
    for (ik, K) in enumerate(K_GRID)
        dseed = SEED_MEANS ? seed_means(Ytr, cond_tr_lag, K, obs_dim) : nothing
        s, el = fit_slds(K, l, Ytr, lds_ws; d_seeds=dseed)
        fitted[(K, l)] = s
        results[ik, jl] = posterior(s, Yva;
            return_γ=false, return_elbo=true,
            progress=true).elbo / Nva
        all_elbos[ik, jl] = el
        println("K = $K  latent = $l   →  held-out ELBO / element = ", round(results[ik, jl]; digits=3))
    end
end

bi = argmax(results);
BEST_K = K_GRID[bi[1]];
BEST_L = LATENT_GRID[bi[2]]
println("\nbest held-out model:  K = $BEST_K,  latent = $BEST_L   (ELBO/element = ",
    round(results[bi]; digits=3), ")")

In [ ]:
if length(K_GRID) > 1 && length(LATENT_GRID) > 1          # full grid → heatmap
    hm = heatmap(string.(LATENT_GRID), string.(K_GRID), results, c=:viridis,
        xlabel="latent dimension", ylabel="discrete states K",
        title="Held-out ELBO / element  (last $N_HELDOUT_SUBJECTS subjects)",
        colorbar_title="ELBO / element")
    for (jl, _) in enumerate(LATENT_GRID), (ik, _) in enumerate(K_GRID)
        annotate!(hm, jl-0.5, ik-0.5, text(round(results[ik, jl]; digits=2), 8, :white))
    end
    scatter!(hm, [bi[2]], [bi[1]], m=(:star5, 12, :red), label="")
    plot(hm, size=(560, 380))
elseif length(LATENT_GRID) == 1                              # sweep K
    plot(K_GRID, results[:, 1], marker=:circle, lw=2, ms=6, xticks=K_GRID,
        xlabel="discrete states K", ylabel="held-out ELBO / element",
        title="latent = $(LATENT_GRID[1])")
else                                                         # sweep latent
    plot(LATENT_GRID, results[1, :], marker=:circle, lw=2, ms=6, xticks=LATENT_GRID,
        xlabel="latent dimension", ylabel="held-out ELBO / element",
        title="K = $(K_GRID[1])")
end

## 6. Pick a fitted model and read its parameters

Everything from here interprets **one** model. By default it is the held-out best from §5, but **override `USE_K` / `USE_L`** to inspect any cell of the grid (e.g. force $K=2$ to look for a story-vs-math split even if the ELBO prefers $K=1$).

The cell below lays out what was fitted: the initial distribution $\pi_k$ and the $K\times K$ discrete transition matrix, each regime's latent dynamics $A_k-I$ and its image in observation space $C_k(A_k-I)C_k^{+}$, the latent offsets $b_k$, and the emission offsets $d_k$.


In [ ]:
USE_K = BEST_K          # ← override to inspect a different model, e.g. USE_K = 2
USE_L = BEST_L          # ←                                    e.g. USE_L = 4
slds = fitted[(USE_K, USE_L)]
println("interpreting SLDS:  K = $USE_K,  latent = $USE_L")



# plot parameters

seq_col = :viridis
div_col = :vik

C1 = slds.LDSs[1].obs_model.C
C2 = slds.LDSs[2].obs_model.C

plt_pi = heatmap(slds.πₖ[:, :], c=seq_col, clims=(0, 1),
     colorbar=false, aspectratio=:equal, ylims=(0.5, USE_K+0.5), xlims=(0.5, 1.5),
     title="p(z_1)")
for k in 1:USE_K
     annotate!(plt_pi, 1, k, text("$(round(slds.πₖ[k]; digits=2))", 12, :white))
end

plt_trans = heatmap(slds.A[:, :], c=seq_col, clims=(0, 1),
     colorbar=false, aspectratio=:equal, lims=(0.5, USE_K+0.5),
     title="discrete transitions")
for j in 1:USE_K
     for k in 1:USE_K
          annotate!(plt_trans, j, k, text("$(round(slds.A[j, k]; digits=2))", 12, :white))

     end
end

plt_A1_lat = heatmap(slds.LDSs[1].state_model.A[:, :] - I, c=div_col, clims=(-0.5, 0.5),
     colorbar=false, aspectratio=:equal, lims=(0.5, USE_L+0.5),
     title="A_1 - I (LDS1  latent)")

plt_A2_lat = heatmap(slds.LDSs[2].state_model.A[:, :] - I, c=div_col, clims=(-0.5, 0.5),
     colorbar=false, aspectratio=:equal, lims=(0.5, USE_L+0.5),
     title="A_2 - I (LDS2 latent)")

plt_A1_obs = heatmap(C1*((slds.LDSs[1].state_model.A[:, :] - I)/C1), c=div_col, clims=(-0.5, 0.5),
     colorbar=false, aspectratio=:equal, lims=(0.5, N_PC+0.5),
     title="A_1 - I (LDS1 obs)")

plt_A2_obs = heatmap(C2*((slds.LDSs[2].state_model.A[:, :] - I)/C2), c=div_col, clims=(-0.5, 0.5),
     colorbar=false, aspectratio=:equal, lims=(0.5, N_PC+0.5),
     title="A_2 - I (LDS2 obs)")

plt_b1 = heatmap(C1*slds.LDSs[1].state_model.b[:, :], c=div_col, clims=(-0.01, 0.01),
     colorbar=false,
     title="b_1 (LDS1 obs)")

plt_b2 = heatmap(C2*slds.LDSs[2].state_model.b[:, :], c=div_col, clims=(-0.01, 0.01),
     colorbar=false,
     title="b_2 (LDS2 obs)")

plt_d1 = heatmap(slds.LDSs[1].obs_model.d[:, :], c=div_col, clims=(-0.1, 0.1),
     colorbar=false,
     title="d_1 (LDS1)")

plt_d2 = heatmap(slds.LDSs[2].obs_model.d[:, :], c=div_col, clims=(-0.1, 0.1),
     colorbar=false,
     title="d_2 (LDS2)")

plot(plt_pi, plt_trans,
     plt_A1_lat, plt_A2_lat,
     plt_A1_obs, plt_A2_obs,
     plt_b1, plt_b2,
     plt_d1, plt_d2,
     layout=(5, 2),
     size=(720, 1440),
)

### Do the regimes differ in mean ($b$) and connectivity ($A$)?

The payoff of an *LDS* over a GLM is that regimes can differ in their **dynamics** $A_k$ (effective connectivity / rotation + decay of the latent), not only their mean. Here we read the fitted model's per-regime dynamics matrices $A_k$ and emission offsets $d_k$ and check that they actually separate — pairwise $\lVert\Delta A\rVert$, the eigenvalue spectra of each $A_k$, and the offset norms $\lVert d_k\rVert$.

In [ ]:
# per-regime dynamics (Aₖ = connectivity) and offsets (dₖ, bₖ = mean) of the interpreted model
As = [slds.LDSs[k].state_model.A for k in 1:USE_K]
bs = [slds.LDSs[k].state_model.b for k in 1:USE_K]
ds = [slds.LDSs[k].obs_model.d for k in 1:USE_K]


println("pairwise regime contrasts:")
for i in 1:USE_K, j in (i+1):USE_K
    println("  states $i,$j:  ‖ΔA‖ = ", round(norm(As[i] - As[j]); digits=3),
        "   ‖Δd‖ = ", round(norm(ds[i] - ds[j]); digits=3),
        "   ‖Δb‖ = ", round(norm(bs[i] - bs[j]); digits=3))
end
for k in 1:USE_K
    ev = eigvals(As[k])
    println("  state $k:  spectral radius = ", round(maximum(abs.(ev)); digits=3),
        "   |λ| angles(rad) = ", round.(sort(abs.(angle.(ev)))[1:min(3, length(ev))]; digits=3))
end

# eigenvalue spectra of each regime's dynamics matrix (dashed unit circle = stability boundary)
θc = range(0, 2π, length=200)
p1 = plot(cos.(θc), sin.(θc), c=:gray, ls=:dash, label="", aspect_ratio=1,
    xlabel="Re(λ)", ylabel="Im(λ)", title="regime dynamics Aₖ eigenvalues")
for k in 1:USE_K
    ev = eigvals(As[k]);
    scatter!(p1, real.(ev), imag.(ev), label="state $k", ms=5)
end
p2 = bar(1:USE_K, [norm(b) for b in bs], legend=false, xticks=1:USE_K,
    xlabel="regime k", ylabel="‖bₖ‖  (latent input offset)", title="regime mean offsets")
plot(p1, p2, size=(820, 340), layout=(1, 2))

## 7. Posterior predictive checks: does the fitted model reproduce the data?

Before asking whether anything in the model lines up with the task, check that it accounts for the signal at all. **Given the frozen parameters, how well does the model predict the BOLD time series?** Two views: predicted vs. observed carpets, and single-channel traces with error bands. §8 and §9 then ask what the continuous latent and the discrete regimes actually *do*.

Everything below rests on one pass of inference with **frozen** parameters. `posterior` returns the regime responsibilities $\gamma_{k,t}$; `smooth(slds, y, \gamma)` then re-runs the Laplace/Kalman smoother under those same weights and hands back both the latent mean $\hat x_t$ and its covariance $V_t$ (`posterior` alone gives $\hat x$ but not $V$). The predicted observation is the responsibility-weighted mixture over regimes,

$$\hat y_t=\sum_k \gamma_{k,t}\,\bigl(C_k\hat x_t+d_k\bigr),$$

carrying **two error bands**, exactly as in **notebook 2**:

- **estimation band** $\;\sum_k\gamma_{k,t}\bigl[\operatorname{diag}(C_kV_tC_k^\top)+(\mu_{k,t}-\hat y_t)^2\bigr]$ — how well the *noise-free readout* is known. The second term is the spread of the per-regime means, i.e. the law of total variance for the mixture over regimes; with $K=1$ it vanishes and this is exactly notebook 2's $CV C^\top$.
- **prediction band** — the same plus $\sum_k\gamma_{k,t}\operatorname{diag}(R_k)$: where a *measured* $y_t$ should land. Its 95% coverage is a calibration check and should come out near 0.95.

**`PRED_SPACE` switches every figure between the `N_PC` observation components and the 100 parcels.** Parcels are reached with the map $R$ that undoes the reduction of §3 (both z-scorings folded in, so it inverts `project` exactly on the retained subspace); $C_k$ and $R_k$ are pushed through it too, rather than rescaling variances after the fact, so the bands stay exact once parcels mix components. Note the honest asymmetry in parcel space: the *observed* trace is the real parcel signal, while the prediction can only live in the `N_PC`-dimensional subspace — the residual therefore contains the variance PCA discarded. The printout reports that ceiling, and §10 reuses the same maps to put the fitted parameters on the cortical surface.


In [ ]:
# ---------------- settings for the prediction figures ----------------
PRED_SET = :val            # runs behind the carpets / traces:  :val | :train | :all
PRED_SPACE = :parcels        # plot in :parcels (Schaefer-100) or :pcs (the N_PC obs dims)
N_PRED_SUBJ = 3               # subjects shown in the carpet / trace figures
N_TRACE_CHAN = 5               # channels per subject in the ribbon figure
TRACE_TRS = 1:min(150, T)   # TR window drawn in the ribbon figure
CARPET_SORT = true            # order parcels by PC-1 loading (makes the carpet legible)
CARPET_RESID = false           # add an observed − predicted residual row

# --- reconstruction (obs→parcel, R) and forward (parcel→obs, F) linear maps ---
# §3 reduces with  s = (Vᵀ(y−pμ)/pσ − sμ)/sσ.  Folding both z-scorings into the loadings
# makes that exactly invertible *on the retained subspace*:  y ≈ R_lin·s + par_off.
# §10 reuses these maps to put the fitted parameters on the cortical surface.
Dp = vec(reducer.pσ)
Ds = vec(reducer.sσ)
Vred = reducer.V                                        # parcels × N_PC
F_lin = Diagonal(1 ./ Ds) * Vred' * Diagonal(1 ./ Dp)   # N_PC × parcels   (parcel→obs)
R_lin = Diagonal(Dp) * pinv(Vred') * Diagonal(Ds)       # parcels × N_PC   (obs→parcel)
par_off = Diagonal(Dp) * (pinv(Vred') * vec(reducer.sμ)) .+ vec(reducer.pμ)   # parcels

# The plot space is just a choice of output map,  out = W_out·obs + off_out.  Cₖ and Rₖ get
# mapped through it below, rather than rescaling variances afterwards: once parcels mix
# components, the obs-space cross-covariances matter and the error bands stay exact.
W_out, off_out, space_name = PRED_SPACE === :parcels ?
                             (R_lin, par_off, "parcel") :
                             (Matrix(1.0I, N_PC, N_PC), zeros(N_PC), "PC")

using DelimitedFiles
parcel_names = readdlm("derivatives/hcp_language_schaefer100_parcels.csv", ',', String; skipstart=1)[:, 2]
# drop the atlas prefix: "17Networks_LH_DefaultA_PFCm_1" → "LH_DefaultA_PFCm_1", so the name fits
# as a panel label
chan_names = PRED_SPACE === :parcels ? replace.(parcel_names, "17Networks_" => "") :
             ["PC $i" for i in 1:N_PC]

println("back-projection ready:  R_lin ", size(R_lin), "   F_lin ", size(F_lin))
println("plotting predictions in $space_name space")

In [ ]:
# ---- runs to predict: model input (obs space) and the signal we score against ----
pred_ep = PRED_SET === :val ? val_ep : PRED_SET === :train ? train_ep : collect(1:E)
subj_pred = subj[pred_ep]
cond_pred = [hrf_shift(cond(e)) for e in pred_ep]

"""
Regime posterior + continuous smoother for a set of runs.  `posterior` returns the
responsibilities γ but not the latent covariance, so we re-run `smooth` with those same
weights: it is the Laplace/Kalman pass the E-step itself uses, and it hands back both
x̂ₜ and Vₜ.  Parameters stay frozen throughout — nothing here refits the model.
"""
function infer_runs(eps; want_cov=true)
    Y = [project(reducer, X[:, :, e]) for e in eps]
    γ = posterior(slds, Y; return_γ=true, return_elbo=false, max_iter=512, progress=true).γ
    xs = Vector{Matrix{Float64}}(undef, length(Y))
    Ps = Vector{Array{Float64,3}}(undef, length(Y))
    for i in eachindex(Y)
        x, V = smooth(slds, Y[i], γ[i])
        xs[i] = copy(x)
        Ps[i] = want_cov ? copy(V) : zeros(0, 0, 0)
    end
    (Y=Y, γ=γ, x=xs, P=Ps)
end

"""
γ-weighted mixture prediction of the observations, in the plot space (`W`, `off`).
Returns the mean and two variances, each `out_dim × T`:

  estimation  Σₖ γₖₜ [ diag(Cₖ Vₜ Cₖᵀ) + (μₖₜ − ŷₜ)² ]   — how well the noise-free readout is known
  predictive  estimation + Σₖ γₖₜ diag(Rₖ)                — where a *measured* yₜ should land

The extra term in the estimation band is the spread of the per-regime means: the law of
total variance for the mixture over regimes.  With K = 1 it vanishes and the two bands
reduce to notebook 2's CVCᵀ and CVCᵀ + R.
"""
function predict_obs(mdl, x, V, γ, W, off)
    K, Tn = size(γ)
    Cs = [W * mdl.LDSs[k].obs_model.C for k in 1:K]                # out × latent
    dk = [W * mdl.LDSs[k].obs_model.d .+ off for k in 1:K]         # out
    rk = [diag(W * mdl.LDSs[k].obs_model.R * W') for k in 1:K]     # out (obs-noise variance)
    μk = [Cs[k] * x .+ dk[k] for k in 1:K]                         # per-regime mean, out × T
    μ = sum(γ[k:k, :] .* μk[k] for k in 1:K)                       # ŷ = E[y]
    sig = zeros(size(μ))
    prd = zeros(size(μ))
    cv = zeros(size(μ))
    for k in 1:K
        @views for t in 1:Tn
            cv[:, t] .= vec(sum((Cs[k] * V[:, :, t]) .* Cs[k], dims=2))   # diag(Cₖ Vₜ Cₖᵀ)
        end
        spread = (μk[k] .- μ) .^ 2
        sig .+= γ[k:k, :] .* (cv .+ spread)
        prd .+= γ[k:k, :] .* (cv .+ spread .+ rk[k])
    end
    (μ=μ, sig=sig, pred=prd)
end

pred = infer_runs(pred_ep)
Opred = PRED_SPACE === :parcels ? [X[:, :, e] for e in pred_ep] : pred.Y   # observed, plot space
nch = size(Opred[1], 1)
pred_run = [predict_obs(slds, pred.x[i], pred.P[i], pred.γ[i], W_out, off_out)
            for i in eachindex(pred.Y)]

fit_r2(O, M) = 1 - sum(abs2, O .- M) / sum(abs2, O .- mean(O, dims=2))
fit_cover(O, M, V) = mean(abs.(O .- M) .<= 2 .* sqrt.(V))
R2_run = [fit_r2(Opred[i], pred_run[i].μ) for i in eachindex(Opred)]
cov_run = [fit_cover(Opred[i], pred_run[i].μ, pred_run[i].pred) for i in eachindex(Opred)]

println("\nprediction on $(length(pred_ep)) runs  (PRED_SET = $PRED_SET, $space_name space):")
println("  R²(y, ŷ) = ", round(mean(R2_run); digits=3), " ± ", round(std(R2_run); digits=3), "  (sd over runs)")
println("  95% predictive coverage = ", round(mean(cov_run); digits=3), "   (nominal 0.954)")
if PRED_SPACE === :parcels     # what the model could reach at best, given the PCA truncation
    ceil_r2 = mean(fit_r2(X[:, :, e], R_lin * project(reducer, X[:, :, e]) .+ par_off) for e in pred_ep)
    println("  ceiling of the $N_PC-component subspace alone: R² = ", round(ceil_r2; digits=3))
end

### Carpets: predicted vs. observed

The same **channel × TR** view as the first look in §2, now with the model's reconstruction underneath each subject's data (one panel per subject, first run, story/math ribbon on top). Both panels are z-scored with the **same — observed — per-channel statistics**, so the predicted carpet is dimmer exactly where the model under-explains instead of being renormalized to look convincing. `CARPET_RESID` adds the residual row; with `CARPET_SORT` the parcels are ordered by their PC-1 loading, which turns the shared low-dimensional structure into visible bands.

In [ ]:
# ---- 1. carpet plots: observed vs. predicted, one column per subject ----
show_subj = unique(subj_pred)[1:min(N_PRED_SUBJ, length(unique(subj_pred)))]
show_i = [findfirst(==(s), subj_pred) for s in show_subj]          # first run of each subject
carpet_ord = (CARPET_SORT && PRED_SPACE === :parcels) ? sortperm(Vred[:, 1]) : collect(1:nch)

# both panels are z-scored with the SAME (observed) per-channel statistics, so the predicted
# carpet is dimmer exactly where the model under-explains rather than being renormalized.
# The residual is already centred, so it is only rescaled — it reads in units of observed SD.
zshow(A, ref; center=true) = (center ? A .- mean(ref, dims=2) : A) ./ (std(ref, dims=2) .+ 1e-8)
carpet(A, ref, lab; center=true) = heatmap(zshow(A, ref; center=center)[carpet_ord, :],
    c=:vik, clims=(-3, 3), colorbar=false, xlabel="TR", ylabel="$lab\n$space_name", titlefontsize=9)

carpet_rib = [cond_ribbon(cond_pred[i];
    title="sub $(subj_pred[i]) · $(runcode[pred_ep[i]] == 0 ? "LR" : "RL")" *
          "   R² = $(round(R2_run[i]; digits=2))") for i in show_i]
carpet_obs = [carpet(Opred[i], Opred[i], "observed") for i in show_i]
carpet_hat = [carpet(pred_run[i].μ, Opred[i], "predicted") for i in show_i]
carpet_res = CARPET_RESID ?
             [carpet(Opred[i] .- pred_run[i].μ, Opred[i], "residual"; center=false) for i in show_i] : []

carpet_rows = CARPET_RESID ? [carpet_rib, carpet_obs, carpet_hat, carpet_res] :
              [carpet_rib, carpet_obs, carpet_hat]
carpet_h = CARPET_RESID ? [0.07, 0.31, 0.31, 0.31] : [0.08, 0.46, 0.46]
plot(reduce(vcat, carpet_rows)...;
    layout=grid(length(carpet_rows), length(show_i), heights=carpet_h),
    size=(360 * length(show_i), 60 + 260 * (length(carpet_rows) - 1)),
    plot_title="observed vs. predicted" *
               (CARPET_SORT && PRED_SPACE === :parcels ? "   ($space_name order = PC-1 loading)" : ""),
    plot_titlefontsize=11)

### Traces with both error bands

Notebook 2's observation-space figure, per subject. The gap between the two bands is the point: the tight **estimation** band says the noise-free readout $C\hat x_t$ is pinned down well; the wide **prediction** band is where a *measured* TR should fall. Scatter of the black observed trace inside the outer band is observation noise, not a failure of the fit — and the printed 95% coverage says whether that claim is calibrated.

In `:pcs` space the channels are simply PCs 1…`N_TRACE_CHAN`. In `:parcels` space they are the parcels with the largest observed **story − math** difference, i.e. the ones the task actually moves; the shading behind each trace is the HRF-lagged block design.

In [ ]:
# ---- 2. traces with both error bands (notebook 2, §"observations reconstructed") ----
# channels: PCs in order; parcels ranked by their observed story−math difference, so the
# panels show the channels the task actually moves rather than an arbitrary slice
function pick_channels(n)
    PRED_SPACE === :pcs && return collect(1:min(n, nch))
    dif = zeros(nch)
    for i in eachindex(Opred)
        s, m = findall(==(1), cond_pred[i]), findall(==(2), cond_pred[i])
        dif .+= vec(mean(Opred[i][:, s], dims=2) .- mean(Opred[i][:, m], dims=2))
    end
    sortperm(abs.(dif); rev=true)[1:min(n, nch)]
end

trace_chan = pick_channels(N_TRACE_CHAN)
println("channels drawn: ", join(["#$c $(chan_names[c])" for c in trace_chan], ",  "))

for i in show_i
    O, pr, tt = Opred[i], pred_run[i], collect(TRACE_TRS)
    rows = map(enumerate(trace_chan)) do (r, ch)
        μ = pr.μ[ch, tt]
        sd_p = 2 .* sqrt.(pr.pred[ch, tt])
        sd_s = 2 .* sqrt.(pr.sig[ch, tt])
        plt = plot(legend=(r == 1 ? :topright : false), legendfontsize=6,
            ylabel=chan_names[ch], ylabelfontsize=6, ytickfontsize=6,
            xlabel=(r == length(trace_chan) ? "TR" : ""), xlim=(first(tt) - 0.5, last(tt) + 0.5))
        c, t = cond_pred[i], first(tt)                    # story/math blocks behind the traces
        while t <= last(tt)
            j = t
            while j < last(tt) && c[j+1] == c[t]
                j += 1
            end
            c[t] == 1 && vspan!(plt, [t - 0.5, j + 0.5], c=:dodgerblue, alpha=0.10, label="")
            c[t] == 2 && vspan!(plt, [t - 0.5, j + 0.5], c=:orange, alpha=0.10, label="")
            t = j + 1
        end
        plot!(plt, tt, μ, ribbon=sd_p, fillalpha=0.15, linealpha=0, c=:crimson, label="±2σ predictive (CVCᵀ+R)")
        plot!(plt, tt, μ, ribbon=sd_s, fillalpha=0.40, linealpha=0, c=:crimson, label="±2σ estimation (CVCᵀ)")
        plot!(plt, tt, O[ch, tt], c=:black, lw=1.2, label="observed y")
        plot!(plt, tt, μ, c=:crimson, lw=1.6, label="predicted ŷ")
        plt
    end
    display(plot(rows...; layout=(length(trace_chan), 1),
        size=(880, 40 + 130 * length(trace_chan)),
        plot_title="sub $(subj_pred[i]) · $(runcode[pred_ep[i]] == 0 ? "LR" : "RL")" *
                   "   R² = $(round(R2_run[i]; digits=2))," *
                   "   95% coverage = $(round(cov_run[i]; digits=2))",
        plot_titlefontsize=10))
end

## 8. The continuous latent at a story ↔ math switch

§7 was about *fit*. This section is about *timing*, and about the **continuous** half of the model: averaged over subjects, what does the latent $\hat x_t$ do around a block switch? All regimes share one latent basis, so $\hat x$ is directly averageable across runs and subjects.

- **Events.** HCP's story and math blocks abut with only 1–2 TRs of fixation between them, so a story onset *is* a math offset: the two event types, `→ story` and `→ math`, cover the switch in both directions. Labels here are deliberately **not** HRF-shifted — the window is drawn relative to the stimulus, so the hemodynamic delay stays visible in the curve rather than being absorbed into the alignment. Averaging is run → subject → group, so every subject counts once regardless of how many usable events their runs contain.
- **Null.** Each run's latent is **circularly rolled** by a random offset and the entire average is recomputed, `EVENT_NULL_N` times. Block structure and the latent's own autocorrelation survive the roll; only the alignment to the task is destroyed. (§9 scores the discrete regimes against the same kind of null.) The grey band is the resulting `EVENT_NULL_CI`% interval, and the printed $p$ compares $\max_t|\text{group mean}|$ against the null distribution of that same statistic, so it needs no further correction across the window.
- **Flags.** `EVENT_GROUP_SE` adds ±1 SEM across subjects; `EVENT_LAT_PCS` rotates the latent onto its own leading principal components and draws that many traces — one shared rotation for all runs, so a PC means the same thing in every panel; set it to `0` to keep the raw latent dimensions (the first `EVENT_MAX_DIM` of them) instead; `EVENT_SET` chooses the subjects (the held-out ones by default; `:all` buys power for the group average, at the cost of the training subjects being in-sample).

A latent whose curve steps outside the grey band around $t=0$ is one whose continuous state is locked to the task.


In [ ]:
# ---- 3. the continuous latent, locked to the story ↔ math switches ----
EVENT_SET = :val         # subjects averaged over:  :all | :val | :train
EVENT_WIN_S = 16.0       # ± window around a block onset, in seconds
EVENT_LAT_PCS = 4        # N > 0 → rotate the latent onto its own first N PCs;  0 → raw latent dims
EVENT_MAX_DIM = 4        # latent dimensions drawn when EVENT_LAT_PCS = 0
EVENT_NULL_N = 1000       # circular-shift null draws (the only slow step here)
EVENT_NULL_CI = 95       # width of the null band, percent

ev_ep = EVENT_SET === :val ? val_ep : EVENT_SET === :train ? train_ep : collect(1:E)
ev_inf = ev_ep == pred_ep ? pred : infer_runs(ev_ep; want_cov=false)   # reuse if same runs
ev_subj = subj[ev_ep]
# UNLAGGED labels here: the window is drawn relative to the *stimulus*, so the hemodynamic
# delay stays visible in the curve instead of being absorbed into the alignment.
ev_cond = [cond(e) for e in ev_ep]

# all regimes share one latent basis, so x̂ is directly averageable across runs.  Centre each
# dimension on its pooled mean (the offset is arbitrary), optionally rotate onto the leading
# latent PCs — the same rotation for every run, so the traces stay comparable across subjects.
ev_mu = mean(reduce(hcat, ev_inf.x), dims=2)
ev_lat = [x .- ev_mu for x in ev_inf.x]
if EVENT_LAT_PCS > 0
    npc = min(EVENT_LAT_PCS, USE_L)
    U = svd(reduce(hcat, ev_lat)).U[:, 1:npc]
    for i in 1:npc
        U[:, i] .*= sign(U[argmax(abs.(U[:, i])), i])    # the sign of a PC is arbitrary
    end
    ev_lat = [U' * x for x in ev_lat]                    # npc × T: one row per PC
    ev_names = ["latent PC $i" for i in 1:npc]
else
    ev_names = ["latent $i" for i in 1:min(EVENT_MAX_DIM, USE_L)]
    ev_lat = [x[1:length(ev_names), :] for x in ev_lat]
end

# A block onset is the first TR whose label differs from the last non-rest label.  HCP's
# story and math blocks abut with only 1–2 TRs of fixation between them, so a story onset is
# also a math offset: these two events cover the switch in both directions.
function block_onsets(c, code)
    on = Int[]
    prev = 0
    for t in eachindex(c)
        c[t] == 0 && continue
        (c[t] == code && prev != code) && push!(on, t)
        prev = c[t]
    end
    on
end

WIN = round(Int, EVENT_WIN_S / TRsec)
ev_lags = collect((-WIN):WIN) .* TRsec
# keep events whose full window fits inside the run; the null then rolls the *latent*, so
# every window stays in bounds and the observed and null averages are scored identically
ev_on = [[[o for o in block_onsets(c, code) if WIN < o <= length(c) - WIN] for c in ev_cond]
         for code in (1, 2)]

"event-locked average of `x` (d × T), with the latent circularly rolled by `shift` (0 = observed)"
function event_avg(x, onsets, W, shift)
    d, Tn = size(x)
    out = zeros(d, 2W + 1)
    @views for o in onsets, (j, τ) in enumerate((-W):W)
        out[:, j] .+= x[:, mod1(o + τ + shift, Tn)]
    end
    out ./ length(onsets)
end

"run → subject → group average, with the SEM across subjects and the subject count"
function group_event(lat, onsets, subjv, W, shifts)
    per = Matrix{Float64}[]
    for s in unique(subjv)
        idx = [i for i in eachindex(lat) if subjv[i] == s && !isempty(onsets[i])]
        isempty(idx) && continue
        push!(per, mean(event_avg(lat[i], onsets[i], W, shifts[i]) for i in idx))
    end
    M = cat(per...; dims=3)
    (mean=dropdims(mean(M, dims=3), dims=3),
        sem=dropdims(std(M, dims=3), dims=3) ./ sqrt(size(M, 3)),
        n=size(M, 3))
end

ev_null = zeros(Int, length(ev_lat))                              # no shift = the observed average
ev_obs = [group_event(ev_lat, ev_on[j], ev_subj, WIN, ev_null) for j in 1:2]

# --- circular-shift null: roll each run's latent by a random offset, redo the whole average.
# Block structure and latent autocorrelation are preserved; only the alignment to the task is
# destroyed — the same logic as the regime scores in §9.
ev_rng = StableRNG(7)
ev_lo = Vector{Matrix{Float64}}(undef, 2)
ev_hi = Vector{Matrix{Float64}}(undef, 2)
ev_p = zeros(length(ev_names), 2)
for j in 1:2
    draws = Array{Float64,3}(undef, size(ev_obs[j].mean)..., EVENT_NULL_N)
    for b in 1:EVENT_NULL_N
        sh = [rand(ev_rng, 1:(size(x, 2)-1)) for x in ev_lat]
        draws[:, :, b] = group_event(ev_lat, ev_on[j], ev_subj, WIN, sh).mean
    end
    α = (100 - EVENT_NULL_CI) / 200
    ev_lo[j] = mapslices(v -> quantile(v, α), draws; dims=3)[:, :, 1]
    ev_hi[j] = mapslices(v -> quantile(v, 1 - α), draws; dims=3)[:, :, 1]
    # family-wise p over the window: max|group mean| against the null of the same statistic,
    # so no correction for the 2·WIN+1 time points is needed
    for d in axes(draws, 1)
        obs_max = maximum(abs, ev_obs[j].mean[d, :])
        ev_p[d, j] = (1 + count(b -> maximum(abs, draws[d, :, b]) >= obs_max, 1:EVENT_NULL_N)) /
                     (EVENT_NULL_N + 1)
    end
end

ev_lab = ["→ story", "→ math"]
println("peri-onset latents:  $(ev_obs[1].n) subjects, $(length(ev_ep)) runs, ",
    "window ±$(round(WIN * TRsec; digits=1)) s, $EVENT_NULL_N circular-shift draws")
println("  events per run:  →story ", round(mean(length.(ev_on[1])); digits=1),
    ",  →math ", round(mean(length.(ev_on[2])); digits=1))
for d in eachindex(ev_names), j in 1:2
    println("  $(ev_names[d])  $(ev_lab[j]):  max|group mean| = ",
        round(maximum(abs, ev_obs[j].mean[d, :]); digits=3),
        "   p(max-stat vs null) = ", round(ev_p[d, j]; digits=3))
end

In [ ]:
EVENT_GROUP_SE = true    # ±1 SEM across subjects around the group mean (display only)

ev_col = [:dodgerblue, :orange]
ev_panels = []
for d in eachindex(ev_names), j in 1:2
    plt = plot(xlabel=(d == length(ev_names) ? "time from block onset (s)" : ""),
        ylabel=(j == 1 ? ev_names[d] : ""), title=(d == 1 ? ev_lab[j] : ""), titlefontsize=9,
        legend=(d == 1 && j == 1 ? :topleft : false), legendfontsize=6)
    mid = (ev_hi[j][d, :] .+ ev_lo[j][d, :]) ./ 2                       # circular-shift null band
    plot!(plt, ev_lags, mid, ribbon=(ev_hi[j][d, :] .- ev_lo[j][d, :]) ./ 2,
        c=:gray, fillalpha=0.35, linealpha=0, label="$EVENT_NULL_CI% null (circular shift)")
    if EVENT_GROUP_SE
        plot!(plt, ev_lags, ev_obs[j].mean[d, :], ribbon=ev_obs[j].sem[d, :],
            c=ev_col[j], fillalpha=0.30, lw=2, label="group mean ± SEM  (n = $(ev_obs[j].n))")
    else
        plot!(plt, ev_lags, ev_obs[j].mean[d, :], c=ev_col[j], lw=2, label="group mean")
    end
    vline!(plt, [0.0], c=:black, ls=:dash, label="")                     # block onset
    hline!(plt, [0.0], c=:gray, ls=:dot, label="")
    push!(ev_panels, plt)
end
plot(ev_panels...; layout=(length(ev_names), 2), size=(880, 150 + 190 * length(ev_names)),
    left_margin=(4, :mm), bottom_margin=(4, :mm),
    plot_title="continuous latent x̂ around story/math block onsets  ($EVENT_SET subjects)",
    plot_titlefontsize=11)

## 9. Do the discrete regimes track the task?

§8 asked the question of the continuous latent; this is the same question asked of the **discrete** state, and it is the one the notebook opened with. We decode the regime posterior $\gamma_{k,t}$ on the held-out runs and draw it **on top of the story/math blocks it never saw**.

`ZHAT_SET` picks which runs to decode (`:val` by default). `DISCRETE_ZHAT` chooses what "the regime" means: with `true` we read off $\hat z_t=\arg\max_k\gamma_{k,t}$ — the SLDS's unsupervised segmentation, one line stepping between regimes. With `false` we keep the posterior itself and plot every $P(z_t=k)$, which shows where the model is genuinely uncertain instead of hiding it behind an argmax. The scores in the next subsection follow the same choice.


In [ ]:
ZHAT_SET = :val            # runs to decode:  :val | :train | :all
DISCRETE_ZHAT = false      # true → the hard segmentation ẑ = argmaxₖ γ;  false → the posterior P(z=k)

if ZHAT_SET == :val
    Y_plt = Yva
    cond_plt = cond_va
    run_plt = runcode[val_ep]
elseif ZHAT_SET == :train
    Y_plt = Ytr
    cond_plt = cond_tr
    run_plt = runcode[train_ep]
elseif ZHAT_SET == :all
    Y_plt = vcat(Ytr, Yva)
    cond_plt = vcat(cond_tr, cond_va)
    run_plt = vcat(runcode[train_ep], runcode[val_ep])
else
    error("ZHAT_SET must be :val, :train, or :all")
end

# HRF-lagged labels were built in §2 (BOLD/ẑ trail the stimulus ~4–6 s → boxcar delayed by HRF_LAG_TR)
println("HRF lag for scoring: $HRF_LAG_TR TRs (", round(HRF_LAG_TR * TRsec; digits=1), " s)")

# regime posteriors on those runs, split back into per-run pieces
γ_ep = posterior(slds, Y_plt;
    return_γ=true, return_elbo=false,
    max_iter=512, progress=true).γ

# Both settings of DISCRETE_ZHAT reduce to one object — a K×T matrix of *responsibilities* per
# run — because the hard segmentation is just the one-hot limit of γ.  The plot and every score
# below then share a single code path, and the flag only decides whether the posterior is
# rounded off before it is read.
function onehot(g)
    m = zeros(size(g))
    for t in axes(g, 2)
        m[argmax(@view g[:, t]), t] = 1.0
    end
    m
end
resp_ep = DISCRETE_ZHAT ? onehot.(γ_ep) : [Matrix{Float64}(g) for g in γ_ep]
zhat_ep = [[argmax(@view g[:, t]) for t in axes(g, 2)] for g in γ_ep]   # hard labels, for reference

# one panel per held-out run: story/math (HRF-lagged) shaded behind the decoded regime
regime_col = [:black, :crimson, :seagreen, :darkviolet, :goldenrod, :teal]

function overlay_panel(r, c, K; title="", leg=false)
    Tn = size(r, 2)
    plt = plot(legend=(leg ? :topright : false), legendfontsize=7,
        title=title, titlefontsize=9, xlabel="TR", xlim=(0.5, Tn + 0.5),
        ylabel=(DISCRETE_ZHAT ? "regime ẑ" : "P(z = k)"),
        yticks=(DISCRETE_ZHAT ? (1:K) : (0:0.5:1)),
        ylim=(DISCRETE_ZHAT ? (0.5, K + 0.5) : (-0.03, 1.03)))
    t = 1                                                # shade contiguous condition blocks
    while t <= Tn
        j = t
        while j < Tn && c[j+1] == c[t]
            j += 1
        end
        c[t] == 1 && vspan!(plt, [t - 0.5, j + 0.5], c=:dodgerblue, alpha=0.16, label="")
        c[t] == 2 && vspan!(plt, [t - 0.5, j + 0.5], c=:orange, alpha=0.16, label="")
        t = j + 1
    end
    if DISCRETE_ZHAT
        plot!(plt, 1:Tn, [argmax(@view r[:, t]) for t in 1:Tn],
            seriestype=:steppost, lw=2, c=:black, label="")
    else
        for k in 1:K
            plot!(plt, 1:Tn, r[k, :], lw=1.6,
                c=regime_col[mod1(k, length(regime_col))], label="P(z=$k)")
        end
    end
    plt
end

nshow = min(6, length(Y_plt))
runlab = ["LR", "RL"]
panels = [overlay_panel(resp_ep[i], cond_plt[i], USE_K;
    title="run $i  ($(runlab[run_plt[i]+1]))", leg=(i == 1)) for i in 1:nshow]
plot(panels..., layout=(nshow, 1), size=(860, 190nshow),
    plot_title=(DISCRETE_ZHAT ? "ẑ" : "P(z = k)") *
               " vs task blocks, $ZHAT_SET runs (HRF-lagged)   (blue = story, orange = math, white = rest)",
    plot_titlefontsize=11)

### Quantifying the overlap

The overlay is qualitative; two summaries make it concrete, pooled over all held-out runs. Both come from **one** object — the soft contingency table

$$W_{kj}\;\propto\;\sum_t \gamma_{k,t}\,\mathbb{1}[c_t=j],$$

the posterior mass regime $k$ places on condition $j$, over task TRs only (rest is dropped). When `DISCRETE_ZHAT = true` the responsibilities are rounded to one-hot first, and $W$ is exactly the count table of $\hat z_t$ against the boxcar — so the hard and soft cases are the *same* statistics evaluated on different inputs, not two different scores. Mutual information and NMI are functionals of the joint distribution, so neither needs re-deriving for a soft assignment.

- **Occupancy $P(\text{regime}\mid\text{condition})$** — $W$ with its columns normalized: the (posterior-weighted) fraction of story / math TRs each regime captures. A regime that specializes for a condition shows up as a bright cell.
- **Mutual information $I(z;c)$**, in bits per TR, with **NMI** $=2I/\big(H(z)+H(c)\big)$ beside it. The two are the same quantity read two ways, and each answers a different question. $I$ is the interpretable one: its ceiling is $H(c)\approx1$ bit — everything there is to know about story-vs-math on a given TR — so $I=0.02$ bits/TR says plainly that the regime posterior carries about 2% of that, which a normalized score in the same situation can flatter. NMI is the comparable one: dividing by $H(z)+H(c)$ removes the mechanical growth of $I$ with $K$, so it is the number to use when reading across the $K$ grid.

Both are compared against a **circular-shift null**: each run's boxcar is randomly rolled, preserving block structure but destroying its alignment to $\gamma$. Scores well above the null mean the regime boundaries track the task beyond what block-autocorrelation alone would give. Note that the roll only permutes each run's labels, so $H(c)$ is identical in every draw — the null moves through $I$ alone — and that $I$ is a plug-in estimate, biased upward by however much freedom $K$ regimes have to carve up the labels; the null carries that same bias, so it cancels out of the reported $z$ and $p$.

In [ ]:
R_all = reduce(hcat, resp_ep)              # K × ΣT responsibilities over the decoded runs, pooled
cc = vcat(cond_plt...)                     # conditions of those runs, HRF-lagged, pooled (0 rest, 1 story, 2 math)
condname = ["story", "math"]
condcode = [1, 2]

# Everything below is read off one object: the soft contingency table W[k, j] — the posterior
# mass regime k places on condition j, over task TRs only.  With one-hot responsibilities it is
# the plain count table of ẑ against the boxcar, so the discrete and soft cases are the same
# statistics rather than two different ones.
ent(p) = -sum(x > 0 ? x * log2(x) : 0.0 for x in p)

"""
Mutual information (bits/TR) and NMI between a regime posterior `R` (K × T) and the HRF-lagged
task labels `c`.  Rest TRs are dropped.  Both are plug-in estimates from the soft table, and
both are functionals of the joint distribution — so a soft `R` needs no different treatment
than a one-hot one.
"""
function overlap_scores(R, c)
    keep = findall(!=(0), c)
    Rk = R[:, keep]
    j = [findfirst(==(cd), condcode) for cd in c[keep]]      # condition index per task TR

    W = zeros(size(R, 1), length(condcode))                  # soft contingency table
    for (i, jj) in enumerate(j), k in axes(Rk, 1)
        W[k, jj] += Rk[k, i]
    end
    W ./= sum(W)

    Pk, Pc = vec(sum(W, dims=2)), vec(sum(W, dims=1))
    mi = sum(W[k, jj] > 0 ? W[k, jj] * log2(W[k, jj] / (Pk[k] * Pc[jj])) : 0.0
             for k in eachindex(Pk), jj in eachindex(Pc))
    d = ent(Pk) + ent(Pc)
    (mi=mi, nmi=(d == 0 ? 0.0 : 2mi / d), Hc=ent(Pc), W=W)
end

obs = overlap_scores(R_all, cc)
occ = obs.W ./ sum(obs.W, dims=1)          # occupancy P(regime | condition): columns of W, normalized

# --- circular-shift null: roll each run's boxcar, keeping block structure but destroying its
# alignment to the regime posterior.  The roll only permutes each run's labels, so H(condition)
# — the ceiling on I, and what you pay guessing from the marginal — is identical in every draw;
# the null moves through I alone.
rng = StableRNG(1)
roll(cs) = vcat([circshift(c, rand(rng, 1:(length(c)-1))) for c in cs]...)
null = [overlap_scores(R_all, roll(cond_plt)) for _ in 1:1000]
mi0 = [s.mi for s in null]
nmi0 = [s.nmi for s in null]
zsc(x, v) = (x - mean(v)) / (std(v) + 1e-12)
pval(x, v) = (1 + count(>=(x), v)) / (length(v) + 1)

println(DISCRETE_ZHAT ? "scoring the hard segmentation ẑ" : "scoring the regime posterior γ (soft)")
println("I(regime; story/math)    = ", round(obs.mi; digits=3), " bits/TR",
    "  of H(condition) = ", round(obs.Hc; digits=3), " bits",
    "   null = ", round(mean(mi0); digits=3), " ± ", round(std(mi0); digits=3),
    "   (z = ", round(zsc(obs.mi, mi0); digits=1), ", p = ", round(pval(obs.mi, mi0); digits=4), ")")
println("NMI (same, normalized)   = ", round(obs.nmi; digits=3),
    "   null = ", round(mean(nmi0); digits=3), " ± ", round(std(nmi0); digits=3),
    "   (z = ", round(zsc(obs.nmi, nmi0); digits=1), ", p = ", round(pval(obs.nmi, nmi0); digits=4), ")")

hm = heatmap(condname, string.(1:USE_K), occ, c=:viridis, clims=(0, 1),
    xlabel="task condition", ylabel="regime k",
    title="P(regime | condition), $ZHAT_SET runs  (K=$USE_K)",
    colorbar_title="occupancy")
for j in 1:2, k in 1:USE_K
    annotate!(hm, j - 0.5, k - 0.5, text(round(occ[k, j]; digits=2), 8, occ[k, j] > 0.5 ? :black : :white))
end
plot(hm, size=(520, 130 + 70USE_K))

## 10. Regimes on the cortical surface — mean ($b$) and connectivity ($A$)

We project each regime's fitted parameters back to the 100 Schaefer parcels and render them on the fsLR-32k surface (`nilearn`, run in a subprocess).

- **Mean map** — each regime's steady-state emission mean $\mu_k = C_k(I-A_k)^{-1}b_k + d_k$ (obs space), lifted to parcels by the reconstruction map $R$ defined in §7, which undoes the PCA/GED reduction *and* the z-scoring. We show each regime and the story−math (state 1 − state 2) difference.
- **Connectivity** — each regime's latent dynamics lifted to a parcel×parcel operator $A^{\mathrm{parcel}}_k = R\,C_k A_k C_k^{+}\,F$ (with $F$ the forward reduction, i.e. $W C A C^{+} W^{+}$ with the z-scoring folded in). From it we plot (1) the first $N$ **principal gradients** — leading **SVD** singular vectors, since $A^{\mathrm{parcel}}$ is non-normal (complex eigenvectors) — and (2) **seed connectivity**: the **efferent** (column) and **afferent** (row) profiles of the parcel with the largest between-state mean difference.

In [ ]:
# R_lin (obs→parcel) and F_lin (parcel→obs) were built in §7, alongside the prediction plots
using MatrixEquations

lat = size(slds.LDSs[1].state_model.A, 1)
Il = Matrix(I, lat, lat)
bmap = zeros(P, USE_K)                                   # parcel-space steady-state mean per regime
Apar = Vector{Matrix{Float64}}(undef, USE_K)            # parcel×parcel connectivity per regime
Apar_sym = Vector{Matrix{Float64}}(undef, USE_K)        # symmetric part of Apar
Apar_rot = Vector{Matrix{Float64}}(undef, USE_K)        # antisymmetric part of Apar
FC = Vector{Matrix{Float64}}(undef, USE_K)              # parcel×parcel steady-state covariance per regime
for k in 1:USE_K
    Ak = slds.LDSs[k].state_model.A;
    bk = slds.LDSs[k].state_model.b
    Ck = slds.LDSs[k].obs_model.C;
    dk = slds.LDSs[k].obs_model.d
    Qk = slds.LDSs[k].state_model.Q;
    Rk = slds.LDSs[k].obs_model.R;
    Pinf = lyapd(Ak, Qk);
    xss = (Il - Ak)\bk                      # steady-state latent (I-A)^{-1} b
    bmap[:, k] = R_lin * (Ck * xss + dk)                 # steady-state mean → parcels
    Apar[k] = R_lin * (Ck * (Ak - I) * pinv(Ck)) * F_lin    # dynamics → parcel×parcel
    Apar_sym[k] = Symmetric(Apar[k] + Apar[k]') .* 0.5
    Apar_rot[k] = (Apar[k] - Apar[k]') .* 0.5
    FC[k] = Symmetric(R_lin * (Ck * Pinf * Ck') * R_lin')
end
println("back-projection ready:  bmap ", size(bmap), "   Apar[1] ", size(Apar[1]))
for k in 1:USE_K
    println("  state $k:  ‖mean map‖ = ", round(norm(bmap[:, k]); digits=3),
        "   ‖A_parcel‖ = ", round(norm(Apar[k]); digits=3))
end

In [ ]:
# steady-state mean map per regime, and the story−math (state 1 − state 2) difference
for k in 1:USE_K
    surfshow(bmap[:, k]; title="state $k · steady-state mean  C(I−A)⁻¹b + d")
end

bmap_d = (bmap[:, 1] .- bmap[:, 2])
USE_K >= 2 && surfshow(bmap_d;
    title="mean difference  (state 1 − state 2)")

In [ ]:
# plot the principal gradients of the effective connectivity and the steady-state covariance, to
# see whether the regimes differ in their dominant modes of interaction.  The SVD is taken
# on the parcel×parcel matrices, so the left singular vectors are the principal gradients.

GRAD_N = 1                                       # number of principal gradients to show
for k in 1:USE_K
    # coactivation
    Ufc = svd(FC[k]).U                            # left singular vectors (real; ordered by gain)
    for n in 1:GRAD_N
        surfshow(Ufc[:, n]; title="state $k · FC principal gradient $n (left SV)", set_vmax=0.4)
    end
    # effective connectivity
    # left singular vectors are the outputs of the effective connectivity
    sv = svd(Apar[k])
    for n in 1:GRAD_N
        surfshow(sv.U[:, n]; title="state $k · effective principal gradient $n (left SV)", set_vmax=0.4)
    end
    # right singular vectors are the inputs of the effective connectivity
    for n in 1:GRAD_N
        surfshow(sv.V[:, n]; title="state $k · effective principal gradient $n (right SV)", set_vmax=0.4)
    end

end

In [ ]:
# plot the symetric part of Apar, which represents the self-connections of each parcel.  The eigenvectors of
# the symmetric part are the principal gradients of the effective connectivity

GRAD_N = 1
for k in 1:USE_K
    # plot the diagonal of the symmetric part of Apar, 
    # which represents the self-connections of each parcel
    surfshow(diag(Apar_sym[k]);
        title="local self-connections of state $k (diag of symetric part of A)",
        set_vmax=0.1
    )

    # plot the eigenvectors of the symmetric part of Apar, which represent the asymptotic
    # directions of the dynamics of the system.  The eigenvectors are ordered by the absolute 
    # value of the eigenvalues.
    ev_sym = eigen(Symmetric(Apar_sym[k]), sortby=real) # symmetric part of Apar
    for n in 1:GRAD_N
        surfshow(ev_sym.vectors[:, n] .* ev_sym.values[n];
            title="state $k · eigvec of symetric part of A $n (SVD)",
            set_vmax=0.1
        )
    end

end

In [ ]:
using DelimitedFiles
parcel_names = readdlm(PARCELS_CSV, ',', String; skipstart=1)[:, 2]

# seed = parcel with the largest between-state mean difference (the strongest encoded parcel)
seed = USE_K >= 2 ? argmax(abs.(bmap[:, 1] .- bmap[:, 2])) : argmax(abs.(bmap[:, 1]))
println("seed parcel #$seed  ($(parcel_names[seed]))")
for k in 1:USE_K
    surfshow(Apar[k][:, seed]; title="state $k · EFFERENT from seed #$seed ($(parcel_names[seed]))", set_vmax=0.01)
end
surfshow(Apar[1][:, seed] .- Apar[2][:, seed]; title="state 1-2 · EFFERENT from seed #$seed ($(parcel_names[seed]))")

for k in 1:USE_K
    surfshow(Apar[k][seed, :]; title="state $k · AFFERENT to seed #$seed ($(parcel_names[seed]))", set_vmax=0.01)
end
surfshow((Apar[1][seed, :] .- Apar[2][seed, :]); title="state 1-2 (diff/sum) · AFFERENT to seed #$seed ($(parcel_names[seed]))")

## 11. Recap & extensions

- We fit the **same SLDS** as Notebook 3 to **real HCP `LANGUAGE` task fMRI**, loaded straight from the parcellation pipeline's `.npz` via `NPZ.jl`, with each **run as a trial** and PCA- (or GED-) reduced parcels as observations.
- Models were compared by **held-out ELBO on unseen subjects** (the last `N_HELDOUT_SUBJECTS`), over a **user-set $K\times\text{latent}$ grid** — set either axis to one value to sweep the other.
- We checked the chosen fit **forward** first: predicted vs. observed carpets and single-channel traces carrying both a $CVC^\top$ **estimation** band and a $CVC^\top+R$ **prediction** band (notebook 2's figure, generalized to the regime mixture), in either parcel or PC space.
- The **continuous latent**, averaged over subjects around story/math onsets, was tested against a **circular-shift null** — asking whether $\hat x_t$ itself is locked to the block boundaries.
- The **discrete regimes** were then laid **directly over the story/math boxcar**, and their overlap quantified by occupancy and by mutual information (in bits/TR, and normalized as NMI) against a **block-preserving circular-shift null** — all read off the same soft contingency table, so they apply to the hard segmentation $\hat z$ and to the raw posterior $\gamma$ alike.
- Each regime's **mean** ($C(I-A)^{-1}b+d$) and **connectivity** ($A^{\mathrm{parcel}}=R\,C A C^{+}\,F$) were projected onto the fsLR-32k Schaefer surface: regime mean maps, $A$ principal gradients (SVD), and seed efferent/afferent connectivity.

**Reading the result honestly.** Whether the SLDS's boundaries snap to the task blocks depends on $K$, latent size, and how strongly story vs. math actually change the *dynamics* (as opposed to the mean signal a GLM would catch). A high NMI-vs-null is real evidence of task-linked switching; a null-level score with a rising held-out ELBO means the regimes capture structure that isn't the block design (subject/run differences, arousal drift, head-motion residual). Both are informative — and the continuous latent in §8 can be task-locked even when the discrete regimes in §9 are not.

**To take further:** sweep `N_PC`, `K_GRID`, `LATENT_GRID`; compare `REPRESENTATION = :zpca` against `:ged`; align $\hat z$ transitions to block *onsets* (a peri-event histogram) rather than pooled occupancy. (Scoring already lag-shifts the boxcar by `HRF_LAG_S` for the hemodynamic delay; set it to 0 to compare.)
